# UVR Model Converter
Converts `.pth` models from the `downloaded/` folder to ONNX format,
then copies them to `android/app/src/main/assets/models/`.

In [1]:
import os
import re
import math
import shutil
import hashlib
import sys

import torch
import torch.onnx

# Ensure the project root is on sys.path so lib_v5 imports work
PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from lib_v5.vr_network import nets, nets_new
from lib_v5.vr_network.model_param_init import ModelParameters

print("Imports OK")
print(f"PyTorch version: {torch.__version__}")

Imports OK
PyTorch version: 2.11.0


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DOWNLOADED_DIR  = os.path.join(PROJECT_ROOT, "downloaded")
MODELPARAMS_DIR = os.path.join(PROJECT_ROOT, "lib_v5", "vr_network", "modelparams")
OUTPUT_ONNX_DIR = os.path.join(PROJECT_ROOT, "onnx_models")          # intermediate staging
ANDROID_ASSETS  = os.path.join(PROJECT_ROOT, "android", "app",
                                "src", "main", "assets", "models")

os.makedirs(OUTPUT_ONNX_DIR, exist_ok=True)
os.makedirs(ANDROID_ASSETS,  exist_ok=True)

print(f"Downloaded dir : {DOWNLOADED_DIR}")
print(f"Staging dir    : {OUTPUT_ONNX_DIR}")
print(f"Android assets : {ANDROID_ASSETS}")

In [ ]:
# ── Helper: resolve model params + primary stem (mirrors UVR.py / separate.py) ─
# UVR identifies a VR model by the MD5 of the last 10 MB of the .pth file and looks
# the hash up in model_data.json to get its modelparams JSON and primary stem.
# Guessing from the filename is wrong for most models (e.g. the SP-UVR models use
# 2band/3band/4band params with 768 bins, not 4band_v3's 672).
import json
from gui_data.constants import secondary_stem

VR_MODEL_DATA_JSON = os.path.join(PROJECT_ROOT, "models", "VR_Models", "model_data", "model_data.json")
with open(VR_MODEL_DATA_JSON) as _f:
    VR_MODEL_DATA = json.load(_f)

def get_model_hash(model_path):
    with open(model_path, 'rb') as f:
        try:
            f.seek(-10000 * 1024, 2)
            return hashlib.md5(f.read()).hexdigest()
        except OSError:
            f.seek(0)
            return hashlib.md5(f.read()).hexdigest()

def get_model_data(model_path):
    """Return UVR's model_data entry ({'vr_model_param', 'primary_stem', ...}) for a .pth."""
    data = VR_MODEL_DATA.get(get_model_hash(model_path))
    if data is None:
        raise KeyError(f"{os.path.basename(model_path)}: hash not in model_data.json — "
                       f"unknown model, cannot determine its parameters")
    return data

def get_model_param_path(model_filename):
    return os.path.join(MODELPARAMS_DIR, f"{get_model_data(model_filename)['vr_model_param']}.json")


# ── Helper: determine nn_arch from file size (mirrors separate.py logic) ──────
NN_ARCH_SIZES   = [31191, 33966, 56817, 123821, 123812, 129605, 218409, 537238, 537227]
VR_51_ARCH      = {56817, 218409}

def get_nn_arch_size(model_path):
    size_kb = math.ceil(os.path.getsize(model_path) / 1024)
    return min(NN_ARCH_SIZES, key=lambda x: abs(x - size_kb))


# ── Helper: load VR model from .pth ───────────────────────────────────────────
def load_vr_model(model_path):
    """
    Load a VR-architecture model (.pth) and return (model, mp, nn_arch_size).
    """
    param_path   = get_model_param_path(model_path)
    mp           = ModelParameters(param_path)
    nn_arch_size = get_nn_arch_size(model_path)
    n_fft        = mp.param['bins'] * 2

    if nn_arch_size in VR_51_ARCH:
        # VR 5.1 architecture (nets_new)
        model = nets_new.CascadedNet(n_fft, nn_arch_size)
    else:
        # Standard VR architecture (nets)
        model = nets.determine_model_capacity(n_fft, nn_arch_size)

    state = torch.load(model_path, map_location="cpu")
    model.load_state_dict(state)
    model.eval()

    return model, mp, nn_arch_size


def build_model_meta(model_path, model, nn_arch_size):
    """Metadata embedded in the .ptl (extra/uvr_meta.json) so the Android app can
    run the same pre/post-processing as desktop UVR without guessing."""
    data = get_model_data(model_path)
    with open(get_model_param_path(model_path)) as f:
        raw_params = json.load(f)
    return {
        "arch": "vr",
        "source_model": os.path.basename(model_path),
        "nn_arch_size": nn_arch_size,
        "is_vr51": nn_arch_size in VR_51_ARCH,
        "offset": model.offset,   # frames cropped from each side of the model output window
        "model_param_name": data["vr_model_param"],
        "model_params": raw_params,
        "primary_stem": data["primary_stem"],
        "secondary_stem": secondary_stem(data["primary_stem"]),
    }


print("Helper functions defined")

In [ ]:
# ── Scan downloaded/ for all .pth files ──────────────────────────────────────
pth_files = []
for root, dirs, files in os.walk(DOWNLOADED_DIR):
    # Skip hidden / system folders
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    for f in files:
        if f.lower().endswith('.pth'):
            pth_files.append(os.path.join(root, f))

pth_files.sort()
print(f"Found {len(pth_files)} .pth file(s):\n")
for p in pth_files:
    size_mb = os.path.getsize(p) / (1024 * 1024)
    param   = os.path.basename(get_model_param_path(p))
    arch    = get_nn_arch_size(p)
    arch_type = "vr51" if arch in VR_51_ARCH else "vr"
    print(f"  {os.path.relpath(p, DOWNLOADED_DIR):<60}  "
          f"{size_mb:6.1f} MB  param={param}  arch={arch}({arch_type})")

In [ ]:
# ── Convert each .pth to TorchScript (mobile-optimised .ptl) ─────────────────
# torch.jit.trace is used because:
#   • Several modules override __call__ instead of forward — trace follows
#     execution regardless of method name.
#   • The HP2 models (arch 537238) share the same conv module object for
#     conv6 and conv7 in ASPPModule.  ONNX mis-handles weight sharing and
#     produces an un-sortable graph; TorchScript trace records the ops
#     correctly.
from torch import nn
from torch.nn.utils.fusion import fuse_conv_bn_eval
from torch.utils.mobile_optimizer import optimize_for_mobile, MobileOptimizerType

OUTPUT_PTL_DIR = os.path.join(PROJECT_ROOT, "ptl_models")   # intermediate staging
os.makedirs(OUTPUT_PTL_DIR, exist_ok=True)

MOBILE_SIZE_LIMIT_MB = 100   # models larger than this are too big for APK bundling
WINDOW_SIZE = 512            # UVR default VR "Window Size" (frames per model call)


def fuse_conv_bn(model):
    """Fold every Conv2d → BatchNorm2d pair (eval mode) into a single Conv2d.

    optimize_for_mobile's own CONV_BN_FUSION pass produces wrong outputs for the
    VR nets (max error ≈ 1.0 on a sigmoid mask), so fusion is done here in eager
    mode and that pass is disabled below.
    """
    for module in model.modules():
        if isinstance(module, nn.Sequential):
            for i in range(len(module) - 1):
                if isinstance(module[i], nn.Conv2d) and isinstance(module[i + 1], nn.BatchNorm2d):
                    module[i] = fuse_conv_bn_eval(module[i], module[i + 1])
                    module[i + 1] = nn.Identity()
    return model


conversion_results = []   # list of (ptl_path, status, message)

for model_path in pth_files:
    model_name = os.path.splitext(os.path.basename(model_path))[0]
    ptl_name   = model_name + "_vr.ptl"
    ptl_path   = os.path.join(OUTPUT_PTL_DIR, ptl_name)
    size_mb    = os.path.getsize(model_path) / (1024 * 1024)

    print(f"\n[{'='*60}]")
    print(f"  Model : {model_name}  ({size_mb:.1f} MB)")

    if size_mb > MOBILE_SIZE_LIMIT_MB:
        print(f"  ⚠  Skipped — {size_mb:.0f} MB exceeds {MOBILE_SIZE_LIMIT_MB} MB APK limit")
        print(f"     Deploy to external storage: Android/data/com.uvr.android/files/models/")
        conversion_results.append((ptl_path, "skipped_large", f"{size_mb:.0f} MB"))
        continue

    try:
        model, mp, nn_arch_size = load_vr_model(model_path)
        bins  = mp.param['bins']
        meta  = build_model_meta(model_path, model, nn_arch_size)
        fuse_conv_bn(model)

        # Dummy input matches what UVR feeds the model: the combined magnitude
        # spectrogram window (batch=1, channels=2, freq=bins+1, time=WINDOW_SIZE).
        # The model returns the mask with the same shape.
        dummy_input = torch.rand(1, 2, bins + 1, WINDOW_SIZE)

        # --- TorchScript trace ------------------------------------------------
        with torch.no_grad():
            traced = torch.jit.trace(model, dummy_input)

        # --- Mobile optimisation (prepacks convs, removes training-only paths) -
        optimized = optimize_for_mobile(
            traced, optimization_blocklist={MobileOptimizerType.CONV_BN_FUSION})

        # torch.jit.save(optimized, ptl_path)
        optimized._save_for_lite_interpreter(
            ptl_path, _extra_files={"uvr_meta.json": json.dumps(meta)})

        ptl_mb = os.path.getsize(ptl_path) / (1024 * 1024)
        print(f"  ✓  Saved  {ptl_name}  ({ptl_mb:.1f} MB)  arch={nn_arch_size}  "
              f"param={meta['model_param_name']}  bins={bins}  primary={meta['primary_stem']}")
        conversion_results.append((ptl_path, "ok", ""))

    except Exception as exc:
        import traceback as _tb
        print(f"  ✗  FAILED: {exc}")
        print(_tb.format_exc())
        conversion_results.append((ptl_path, "failed", str(exc)))
        

print("\n" + "="*64)
ok      = sum(1 for _, s, _ in conversion_results if s == "ok")
skipped = sum(1 for _, s, _ in conversion_results if s.startswith("skipped"))
failed  = sum(1 for _, s, _ in conversion_results if s == "failed")
print(f"Conversion complete: {ok} succeeded, {skipped} skipped (too large), {failed} failed.")


In [ ]:
# ── Validate TorchScript models against the original .pth ─────────────────────
print("Validating TorchScript (.ptl) models...\n")

for ptl_path, status, msg in conversion_results:
    if status != "ok":
        tag = "⚠  large" if status.startswith("skipped") else "✗  failed"
        print(f"  {tag}  {os.path.basename(ptl_path)}  ({msg})")
        continue

    try:
        extra  = {"uvr_meta.json": ""}
        loaded = torch.jit.load(ptl_path, map_location="cpu", _extra_files=extra)
        loaded.eval()
        meta = json.loads(extra["uvr_meta.json"])

        model_stem = os.path.basename(ptl_path).replace("_vr.ptl", "")
        orig_path  = next(p for p in pth_files
                          if os.path.splitext(os.path.basename(p))[0] == model_stem)
        orig, mp, _ = load_vr_model(orig_path)
        bins = mp.param['bins']

        x = torch.rand(1, 2, bins + 1, WINDOW_SIZE)
        with torch.no_grad():
            out, ref = loaded(x), orig(x)
        err = (out - ref).abs().max().item()

        print(f"  ✓  {os.path.basename(ptl_path)}  →  output {tuple(out.shape)}  "
              f"max|ptl-pth|={err:.2e}  param={meta['model_param_name']}")

    except Exception as exc:
        print(f"  ✗  {os.path.basename(ptl_path)}: {exc}")


In [ ]:
# ── Convert each .pth to TFLite FP32 for the Android GPU delegate ─────────────
# The app runs <name>_vr.tflite (beside <name>_vr.ptl) on the GPU in full FP32:
# ~3x faster than the .ptl on CPU and closer to the .pth (≈1e-5 vs ≈1e-4).
# The .ptl stays the source of uvr_meta.json and the CPU fallback / reference.
#
# Path: fused eager model → ONNX → onnxsim → onnx2tf. onnx2tf needs TensorFlow,
# which the base env can't import (numpy too old), so those steps run in a
# separate conda env, created once with:
#   conda create -n uvr-convert python=3.11
#   conda run -n uvr-convert pip install tensorflow==2.17.1 tf_keras==2.17.0 onnx==1.16.2 \
#       onnx2tf==1.26.3 onnx_graphsurgeon sng4onnx onnxsim psutil "numpy<2" ai-edge-litert
import subprocess, tempfile
import numpy as np
from lib_v5.vr_network import layers

CONVERT_ENV = "uvr-convert"
TFLITE_MAX_ERR = 5e-4   # vs the .pth; FP32 noise is 1e-5–2e-4 depending on the model, FP16 would be ~0.1


def aspp_forward_broadcast(self, x):
    """ASPPModule.forward with feat1's bilinear resize from 1 row to h rows
    (align_corners, same width) written as the broadcast it is. Identical
    output; onnx2tf can't place the 1-row Resize in NHWC otherwise."""
    _, _, h, w = x.size()
    feats = [self.conv1(x).expand(-1, -1, h, w), self.conv2(x), self.conv3(x), self.conv4(x), self.conv5(x)]
    if self.nn_architecture in self.six_layer:
        feats.append(self.conv6(x))
    elif self.nn_architecture in self.seven_layer:
        feats += [self.conv6(x), self.conv7(x)]
    return self.bottleneck(torch.cat(feats, dim=1))


VALIDATE_TFLITE = """
import sys, numpy as np
from ai_edge_litert.interpreter import Interpreter
it = Interpreter(sys.argv[1], num_threads=4); it.allocate_tensors()
x = np.load(sys.argv[2]); ref = np.load(sys.argv[3])
it.set_tensor(it.get_input_details()[0]['index'], x.transpose(0, 2, 3, 1).copy()); it.invoke()
y = it.get_tensor(it.get_output_details()[0]['index']).transpose(0, 3, 1, 2)
print(np.abs(y - ref).max())
"""

tflite_results = []
for ptl_path, status, _ in conversion_results:
    if status != "ok":
        continue
    stem = os.path.basename(ptl_path).replace("_vr.ptl", "")
    model_path = next(p for p in pth_files if os.path.splitext(os.path.basename(p))[0] == stem)
    tfl_path = ptl_path[:-len(".ptl")] + ".tflite"
    try:
        model, mp, _ = load_vr_model(model_path)
        bins = mp.param['bins']
        x = torch.rand(1, 2, bins + 1, WINDOW_SIZE, generator=torch.Generator().manual_seed(0)) ** 4
        with torch.no_grad():
            ref = model(x).numpy()
        fuse_conv_bn(model)

        work = tempfile.mkdtemp()
        original_forward = layers.ASPPModule.forward
        layers.ASPPModule.forward = aspp_forward_broadcast
        try:
            with torch.no_grad():
                torch.onnx.export(model, x, os.path.join(work, "m.onnx"), input_names=["input"],
                                  output_names=["output"], opset_version=17)
        finally:
            layers.ASPPModule.forward = original_forward
        np.save(os.path.join(work, "x.npy"), x.numpy())
        np.save(os.path.join(work, "ref.npy"), ref)

        def run_in_env(*cmd):
            return subprocess.run(["conda", "run", "-n", CONVERT_ENV, *cmd], cwd=work,
                                  check=True, capture_output=True, text=True).stdout
        run_in_env("onnxsim", "m.onnx", "ms.onnx")
        run_in_env("onnx2tf", "-i", "ms.onnx", "-o", "tf", "-coion")
        converted = os.path.join(work, "tf", "ms_float32.tflite")
        err = float(run_in_env("python", "-c", VALIDATE_TFLITE, converted, "x.npy", "ref.npy").split()[-1])
        if err > TFLITE_MAX_ERR:
            raise ValueError(f"max|tflite-pth| = {err:.2e} > {TFLITE_MAX_ERR}")
        shutil.copy(converted, tfl_path)
        shutil.rmtree(work)
        print(f"  ✓  {os.path.basename(tfl_path)}  max|tflite-pth|={err:.2e}")
        tflite_results.append((tfl_path, "ok", ""))
    except Exception as exc:
        print(f"  ✗  {os.path.basename(tfl_path)}: {exc}")
        tflite_results.append((tfl_path, "failed", str(exc)))


In [ ]:
# ── Validate TorchScript models ───────────────────────────────────────────────
print("Validating TorchScript (.ptl) models...\n")

for ptl_path, status, msg in conversion_results:
    if status != "ok":
        tag = "⚠  large" if status.startswith("skipped") else "✗  failed"
        print(f"  {tag}  {os.path.basename(ptl_path)}  ({msg})")
        continue

    try:
        loaded = torch.jit.load(ptl_path, map_location="vulkan")
        loaded.eval()

        # Infer bins from the ptl filename (same pattern as conversion)
        model_stem = os.path.basename(ptl_path).replace("_vr.ptl", "")
        orig_path  = next((p for p in pth_files
                           if os.path.splitext(os.path.basename(p))[0] == model_stem), None)
        if orig_path:
            mp   = ModelParameters(get_model_param_path(orig_path))
            bins = mp.param['bins']
        else:
            bins = 1024   # safe fallback

        dummy = torch.zeros(1, 2, bins, 128)
        with torch.no_grad():
            out = loaded(dummy)

        print(f"  ✓  {os.path.basename(ptl_path)}  →  output shape {tuple(out.shape)}")

    except Exception as exc:
        print(f"  ✗  {os.path.basename(ptl_path)}: {exc}")


In [ ]:
# ── Copy .ptl files to Android assets ────────────────────────────────────────
print(f"Copying TorchScript models to:\n  {ANDROID_ASSETS}\n")

copied  = []
skipped = []

for ptl_path, status, _ in conversion_results + tflite_results:
    if status != "ok":
        skipped.append(ptl_path)
        continue

    dest = os.path.join(ANDROID_ASSETS, os.path.basename(ptl_path))

    # Avoid re-copying if already identical
    if os.path.isfile(dest):
        def md5(path):
            h = hashlib.md5()
            with open(path, "rb") as f:
                for chunk in iter(lambda: f.read(65536), b""):
                    h.update(chunk)
            return h.hexdigest()

        if md5(ptl_path) == md5(dest):
            print(f"  =  (unchanged) {os.path.basename(ptl_path)}")
            skipped.append(ptl_path)
            continue

    shutil.copy2(ptl_path, dest)
    size_mb = os.path.getsize(dest) / (1024 * 1024)
    print(f"  ✓  Copied  {os.path.basename(ptl_path)}  ({size_mb:.1f} MB)")
    copied.append(dest)

print(f"\nDone: {len(copied)} file(s) copied, {len(skipped)} skipped.")
print(f"\nAndroid assets directory contents:")
for f in sorted(os.listdir(ANDROID_ASSETS)):
    fp = os.path.join(ANDROID_ASSETS, f)
    size_mb = os.path.getsize(fp) / (1024 * 1024)
    print(f"  {f:<65}  {size_mb:6.1f} MB")


In [ ]:
# ── Export Demucs v4 models (htdemucs, htdemucs_6s, htdemucs_ft) for Android ─────
# The app runs HTDemucs's STFT/iSTFT and desktop's apply_model/demix_demucs itself;
# only the network core is exported (demucs_onnx.HTDemucsCore). Each model becomes a
# <name>.demucs manifest plus one <name>_<i>.onnx per sub-model of the bag
# (htdemucs_ft has 4). Copy all of them into the app's models folder together.
# MDX-Net runs UVR's original .onnx files (settings found by model hash, as on
# desktop); the next cell adds the optional GPU versions.
from demucs_onnx import export_demucs

DEMUCS_REPO = os.path.join(PROJECT_ROOT, "models", "Demucs_Models", "v3_v4_repo")
for demucs_name in ["htdemucs", "htdemucs_6s", "htdemucs_ft"]:
    if not os.path.exists(os.path.join(DEMUCS_REPO, f"{demucs_name}.yaml")):
        print(f"  ⚠  {demucs_name}: not downloaded")
        continue
    for f, rel_err in export_demucs(demucs_name, DEMUCS_REPO, OUTPUT_ONNX_DIR):
        print(f"  ✓  {f}  max|onnx-torch| = {rel_err:.1e} of peak")


In [ ]:
# ── MDX-Net .tflite for the Android GPU delegate (optional) ─────────────────────
# The app runs <name>.tflite beside UVR's <name>.onnx on the GPU in FP16 when that is
# faster than the CPU and the first non-silent chunk matches it (Adreno 650: ~2.4x
# per chunk, SDR within 0.01 dB). Copy both files into the app's models folder.
# Needs the uvr-convert env from the VR TFLite cell; ~1 min and 67 MB per model.
MDX_DIR = os.path.join(PROJECT_ROOT, "models", "MDX_Net_Models")
for mdx_name in ["Kim_Vocal_2"]:
    src = os.path.join(MDX_DIR, f"{mdx_name}.onnx")
    if not os.path.exists(src):
        print(f"  ⚠  {mdx_name}: not downloaded")
        continue
    dest = os.path.join(OUTPUT_ONNX_DIR, f"{mdx_name}.tflite")
    out = subprocess.run(["conda", "run", "-n", CONVERT_ENV, "python", os.path.join(PROJECT_ROOT, "mdx_tflite.py"), src, dest],
                         capture_output=True, text=True)
    if out.returncode:
        print(f"  ✗  {mdx_name}.tflite: {out.stderr.strip().splitlines()[-1]}")
    else:
        print(f"  ✓  {mdx_name}.tflite  max|tflite-onnx| = {float(out.stdout.split()[-1]):.1e} of peak")
